Install required libraries

In [12]:
!pip install gradio pandas

ICT Helpdesk application

In [24]:
import gradio as gr
import pandas as pd
from datetime import datetime

tickets = []


# 1. Submit Complaint


def submit_ticket(name, role, department, category, description):

    if not name.strip() or not description.strip():
        return "Please enter your name and problem description.", "", gr.update()

    # System Ticket ID
    ticket_id = "ICT" + str(len(tickets) + 1001)

    # Save Complaint
    tickets.append({
        "Ticket ID": ticket_id,
        "Name": name,
        "Role": role,
        "Department": department,
        "Category": category,
        "Description": description,
        "Status": "Pending",
        "Created At": datetime.now().strftime("%Y-%m-%d %H:%M")
    })

    # Automatically update Ticket ID list
    ticket_ids = [ticket["Ticket ID"] for ticket in tickets]

    return (
        f"Complaint submitted successfully! Ticket ID: {ticket_id}",
        ticket_id,
        gr.update(choices=ticket_ids, value=ticket_id)
    )



# 2. View Complaints


def view_tickets():

    if not tickets:
        return pd.DataFrame(columns=[
            "Ticket ID", "Name", "Role", "Department",
            "Category", "Description", "Status", "Created At"
        ])

    return pd.DataFrame(tickets)


# 3. Update Status


def update_status(ticket_id, status):

    if not ticket_id:
        return "Please select a Ticket ID.", view_tickets()

    for ticket in tickets:

        if ticket["Ticket ID"] == ticket_id:
            ticket["Status"] = status

            return (
                f"{ticket_id} status updated to {status}.",
                view_tickets()
            )

    return "Ticket ID not found.", view_tickets()



# Gradio Interface


with gr.Blocks(title="Campus ICT Helpdesk") as demo:

    gr.Markdown("# Campus ICT Helpdesk System")
    gr.Markdown("College & University IT Support Portal")



    # Submit Complaint


    with gr.Tab("Submit Complaint"):

        name = gr.Textbox(
            label="Student/Teacher Name"
        )

        role = gr.Dropdown(
            ["Student", "Teacher", "Staff"],
            label="Your Role"
        )

        department = gr.Textbox(
            label="Department"
        )

        category = gr.Dropdown(
            [
                "Wi-Fi/Internet",
                "Computer Problem",
                "Projector Problem",
                "Login Problem",
                "Other"
            ],
            label="Problem Category"
        )

        description = gr.Textbox(
            label="Describe Your Problem",
            lines=4
        )

        submit = gr.Button(
            "Submit Complaint",
            variant="primary"
        )

        result = gr.Textbox(
            label="Result"
        )

        ticket_output = gr.Textbox(
            label="Your Ticket ID"
        )



    # View Complaints


    with gr.Tab("View Complaints"):

        refresh = gr.Button(
            "Refresh Complaint List"
        )

        table = gr.Dataframe(
            interactive=False
        )

        refresh.click(
            view_tickets,
            outputs=table
        )



    # Admin Panel

    with gr.Tab("Admin Panel"):

        admin_ticket_id = gr.Dropdown(
            choices=[],
            label="Select Ticket ID"
        )

        new_status = gr.Dropdown(
            [
                "Pending",
                "In Progress",
                "Resolved"
            ],
            label="New Status",
            value="In Progress"
        )

        update_button = gr.Button(
            "Update Status"
        )

        admin_result = gr.Textbox(
            label="Update Result"
        )

        update_button.click(
            update_status,
            inputs=[
                admin_ticket_id,
                new_status
            ],
            outputs=[
                admin_result,
                table
            ]
        )


    # Submit button event
    submit.click(
        submit_ticket,
        inputs=[
            name,
            role,
            department,
            category,
            description
        ],
        outputs=[
            result,
            ticket_output,
            admin_ticket_id
        ]
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ba3dc56030d17673d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
